# EQL-Abfragen auf einer Plan-Datenbank

Fragt einen aufgezeichneten Roboterplan mit EQL ab. Für jede Abfrage zeigt das Notebook die EQL-Abfrage, das vom Übersetzer generierte SQL und das Ergebnis.

Die Datenbank wird mit `plan_db_create.ipynb` erzeugt. Dieses Notebook braucht einen eigenen Kernel: Die Plan-ORM-Klassen und die Testklassen aus `metrics_evaluation.ipynb` registrieren Tabellen in denselben SQLAlchemy-Metadaten und lassen sich nicht gemeinsam laden.

## Setup

In [1]:
import os
from collections import Counter

from IPython.display import Markdown, display
from sqlalchemy.orm import sessionmaker

import coraplex.orm.ormatic_interface as coraplex_orm
from coraplex.datastructures.enums import Arms
from krrood.entity_query_language.factories import entity, variable, an, set_of
from krrood.ormatic.eql_interface import eql_to_sql
from krrood.ormatic.utils import create_engine

PLAN_DATABASE_PATH = "/home/jovyan/work/tutorials/plan_result_rich.sqlite"

if not os.path.exists(PLAN_DATABASE_PATH):
    raise FileNotFoundError(f"{PLAN_DATABASE_PATH} not found. Run plan_db_create.ipynb first.")

plan_session = sessionmaker(bind=create_engine(f"sqlite+pysqlite:///{PLAN_DATABASE_PATH}"))()


def values(row):
    """Short text for a set_of() result row: its values, enums by name."""
    return ", ".join(getattr(v, "name", None) or str(v) for v in row.values())


def run_query(title, query, eql_text, row_format=values, max_rows=10):
    """Show EQL, generated SQL and result; identical rows are counted."""
    translator = eql_to_sql(query, plan_session)
    display(Markdown(
        f"### {title}\n\n"
        f"**EQL**\n```python\n{eql_text.strip()}\n```\n\n"
        f"**Generated SQL**\n```sql\n{str(translator.sql_query).strip()}\n```"
    ))
    rows = [row_format(row) for row in translator.evaluate()]
    counts = Counter(rows)
    print(f"{len(rows)} result(s)")
    if len(counts) < len(rows):
        for row, n in counts.most_common():
            print(f"  {n:>3} x {row}")
    else:
        for row in rows[:max_rows]:
            print(f"  {row}")
        if len(rows) > max_rows:
            print(f"  ... ({len(rows) - max_rows} more)")

## 1. Welcher Arm wurde benutzt?

Der Arm jeder Transportaktion: ein einzelnes Attribut.

In [2]:
action = variable(type_=coraplex_orm.TransportActionDAO, domain=[])
run_query(
    "Which arm was used?",
    an(set_of(action.arm)),
    "an(set_of(action.arm))",
)

### Which arm was used?

**EQL**
```python
an(set_of(action.arm))
```

**Generated SQL**
```sql
SELECT "TransportActionDAO".arm 
FROM "DesignatorDAO" JOIN "ActionDescriptionDAO" ON "ActionDescriptionDAO".database_id = "DesignatorDAO".database_id JOIN "TransportActionDAO" ON "TransportActionDAO".database_id = "ActionDescriptionDAO".database_id
```

3 result(s)
    2 x LEFT
    1 x RIGHT


## 2. Welches Objekt wurde mit welchem Arm transportiert?

`action.object_designator.name.name` folgt zwei Beziehungen, ohne dass ein Fremdschlüssel angegeben wird.

In [3]:
action = variable(type_=coraplex_orm.TransportActionDAO, domain=[])
run_query(
    "Objects per arm",
    an(set_of(action.arm, action.object_designator.name.name)),
    "an(set_of(action.arm, action.object_designator.name.name))",
)

### Objects per arm

**EQL**
```python
an(set_of(action.arm, action.object_designator.name.name))
```

**Generated SQL**
```sql
SELECT "TransportActionDAO".arm, "PrefixedNameDAO_1".name 
FROM "DesignatorDAO" JOIN "ActionDescriptionDAO" ON "ActionDescriptionDAO".database_id = "DesignatorDAO".database_id JOIN "TransportActionDAO" ON "TransportActionDAO".database_id = "ActionDescriptionDAO".database_id JOIN ("WorldEntityDAO" AS "WorldEntityDAO_1" JOIN "WorldEntityWithIDDAO" AS "WorldEntityWithIDDAO_1" ON "WorldEntityWithIDDAO_1".database_id = "WorldEntityDAO_1".database_id JOIN "WorldEntityWithSimulatorPropertiesDAO" AS "WorldEntityWithSimulatorPropertiesDAO_1" ON "WorldEntityWithSimulatorPropertiesDAO_1".database_id = "WorldEntityWithIDDAO_1".database_id JOIN "SemanticAnnotationDAO" AS "SemanticAnnotationDAO_1" ON "SemanticAnnotationDAO_1".database_id = "WorldEntityWithSimulatorPropertiesDAO_1".database_id JOIN "HasRootKinematicStructureEntityDAO" AS "HasRootKinematicStructureEntityDAO_1" ON "HasRootKinematicStructureEntityDAO_1".database_id = "SemanticAnnotationDAO_1".database_id JOIN "HasRootBodyDAO" AS "HasRootBodyDAO_1" ON "HasRootBodyDAO_1".database_id = "HasRootKinematicStructureEntityDAO_1".database_id) ON "HasRootBodyDAO_1".database_id = "TransportActionDAO"._object_designator_id JOIN "PrefixedNameDAO" AS "PrefixedNameDAO_1" ON "PrefixedNameDAO_1".database_id = "WorldEntityDAO_1"._name_id
```

3 result(s)
  LEFT, Bowl
  LEFT, Spoon
  RIGHT, Milk


## 3. Transportaktionen mit dem linken Arm

Die Bedingung liest sich wie Python: `action.arm == Arms.LEFT`. Die Abfrage liefert vollständige Aktionen; ausgegeben werden nur Arm und Objekt.

In [4]:
action = variable(type_=coraplex_orm.TransportActionDAO, domain=[])
run_query(
    "Transport actions with the left arm",
    an(entity(action).where(action.arm == Arms.LEFT)),
    "an(entity(action).where(action.arm == Arms.LEFT))",
    row_format=lambda a: f"TransportAction(arm={a.arm.name}, object={a.object_designator.name.name})",
)

### Transport actions with the left arm

**EQL**
```python
an(entity(action).where(action.arm == Arms.LEFT))
```

**Generated SQL**
```sql
SELECT "TransportActionDAO".database_id, "ActionDescriptionDAO".database_id AS database_id_1, "DesignatorDAO".database_id AS database_id_2, "DesignatorDAO".polymorphic_type, "TransportActionDAO".arm, "TransportActionDAO"._object_designator_id, "TransportActionDAO"._target_location_id, "TransportActionDAO"._grasp_description_id 
FROM "DesignatorDAO" JOIN "ActionDescriptionDAO" ON "ActionDescriptionDAO".database_id = "DesignatorDAO".database_id JOIN "TransportActionDAO" ON "TransportActionDAO".database_id = "ActionDescriptionDAO".database_id 
WHERE "TransportActionDAO".arm = :arm_1
```

2 result(s)
  TransportAction(arm=LEFT, object=Bowl)
  TransportAction(arm=LEFT, object=Spoon)


## 4. Status der Aktionsknoten

Der Status jedes Aktionsknotens im Plan (`ActionNodeDAO`).

In [5]:
node = variable(type_=coraplex_orm.ActionNodeDAO, domain=[])
run_query(
    "Status of the action nodes",
    an(set_of(node.status)),
    "an(set_of(node.status))",
)

### Status of the action nodes

**EQL**
```python
an(set_of(node.status))
```

**Generated SQL**
```sql
SELECT "PlanNodeDAO".status 
FROM "PlanEntityDAO" JOIN "PlanNodeDAO" ON "PlanNodeDAO".database_id = "PlanEntityDAO".database_id JOIN "DesignatorNodeDAO" ON "DesignatorNodeDAO".database_id = "PlanNodeDAO".database_id JOIN "ActionNodeDAO" ON "ActionNodeDAO".database_id = "DesignatorNodeDAO".database_id
```

35 result(s)
   35 x NOT_STARTED


## 5. Welche Arten von Aktionen kommen im Plan vor?

Die Art der Aktion steht in einer Vererbungstabelle (`polymorphic_type`); in EQL erreicht man sie über `node.designator`.

In [6]:
node = variable(type_=coraplex_orm.ActionNodeDAO, domain=[])
run_query(
    "Kinds of actions",
    an(set_of(node.status, node.designator.polymorphic_type)),
    "an(set_of(node.status, node.designator.polymorphic_type))",
)

### Kinds of actions

**EQL**
```python
an(set_of(node.status, node.designator.polymorphic_type))
```

**Generated SQL**
```sql
SELECT "PlanNodeDAO".status, "DesignatorDAO_1".polymorphic_type 
FROM "PlanEntityDAO" JOIN "PlanNodeDAO" ON "PlanNodeDAO".database_id = "PlanEntityDAO".database_id JOIN "DesignatorNodeDAO" ON "DesignatorNodeDAO".database_id = "PlanNodeDAO".database_id JOIN "ActionNodeDAO" ON "ActionNodeDAO".database_id = "DesignatorNodeDAO".database_id JOIN "DesignatorDAO" AS "DesignatorDAO_1" ON "DesignatorDAO_1".database_id = "DesignatorNodeDAO"._designator_id
```

35 result(s)
   10 x NOT_STARTED, ParkArmsActionDAO
    7 x NOT_STARTED, NavigateActionDAO
    4 x NOT_STARTED, MoveTorsoActionDAO
    3 x NOT_STARTED, TransportActionDAO
    3 x NOT_STARTED, PickUpActionDAO
    3 x NOT_STARTED, PlaceActionDAO
    3 x NOT_STARTED, ReachActionDAO
    1 x NOT_STARTED, OpenActionDAO
    1 x NOT_STARTED, GraspingActionDAO


## Eigene Abfrage

In [7]:
action = variable(type_=coraplex_orm.TransportActionDAO, domain=[])
run_query(
    "Ob5. Welche Arten von Aktionen kommen im Plan vor?
Die Art der Aktion steht in einer Vererbungstabelle (polymorphic_type); in EQL erreicht man sie über node.designator.

node = variable(type_=coraplex_orm.ActionNodeDAO, domain=[])
run_query(
    "Kinds of actions",
    an(set_of(node.status, node.designator.polymorphic_type)),
    "an(set_of(node.status, node.designator.polymorphic_type))",
)
Kinds of actions
EQL

an(set_of(node.status, node.designator.polymorphic_type))
Generated SQL

SELECT "PlanNodeDAO".status, "DesignatorDAO_1".polymorphic_type 
FROM "PlanEntityDAO" JOIN "PlanNodeDAO" ON "PlanNodeDAO".database_id = "PlanEntityDAO".database_id JOIN "DesignatorNodeDAO" ON "DesignatorNodeDAO".database_id = "PlanNodeDAO".database_id JOIN "ActionNodeDAO" ON "ActionNodeDAO".database_id = "DesignatorNodeDAO".database_id JOIN "DesignatorDAO" AS "DesignatorDAO_1" ON "DesignatorDAO_1".database_id = "DesignatorNodeDAO"._designator_id
35 result(s)
   10 x NOT_STARTED, ParkArmsActionDAO
    7 x NOT_STARTED, NavigateActionDAO
    4 x NOT_STARTED, MoveTorsoActionDAO
    3 x NOT_STARTED, TransportActionDAO
    3 x NOT_STARTED, PickUpActionDAO
    3 x NOT_STARTED, PlaceActionDAO
    3 x NOT_STARTED, ReachActionDAO
    1 x NOT_STARTED, OpenActionDAO
    1 x NOT_STARTED, GraspingActionDAO






jects transported with the right arm",
    an(set_of(action.object_designator.name.name).where(action.arm == Arms.RIGHT)),
    "an(set_of(action.object_designator.name.name).where(action.arm == Arms.RIGHT))",
)

### Objects transported with the right arm

**EQL**
```python
an(set_of(action.object_designator.name.name).where(action.arm == Arms.RIGHT))
```

**Generated SQL**
```sql
SELECT "PrefixedNameDAO_1".name 
FROM "DesignatorDAO" JOIN "ActionDescriptionDAO" ON "ActionDescriptionDAO".database_id = "DesignatorDAO".database_id JOIN "TransportActionDAO" ON "TransportActionDAO".database_id = "ActionDescriptionDAO".database_id JOIN ("WorldEntityDAO" AS "WorldEntityDAO_1" JOIN "WorldEntityWithIDDAO" AS "WorldEntityWithIDDAO_1" ON "WorldEntityWithIDDAO_1".database_id = "WorldEntityDAO_1".database_id JOIN "WorldEntityWithSimulatorPropertiesDAO" AS "WorldEntityWithSimulatorPropertiesDAO_1" ON "WorldEntityWithSimulatorPropertiesDAO_1".database_id = "WorldEntityWithIDDAO_1".database_id JOIN "SemanticAnnotationDAO" AS "SemanticAnnotationDAO_1" ON "SemanticAnnotationDAO_1".database_id = "WorldEntityWithSimulatorPropertiesDAO_1".database_id JOIN "HasRootKinematicStructureEntityDAO" AS "HasRootKinematicStructureEntityDAO_1" ON "HasRootKinematicStructureEntityDAO_1".database_id = "SemanticAnnotationDAO_1".database_id JOIN "HasRootBodyDAO" AS "HasRootBodyDAO_1" ON "HasRootBodyDAO_1".database_id = "HasRootKinematicStructureEntityDAO_1".database_id) ON "HasRootBodyDAO_1".database_id = "TransportActionDAO"._object_designator_id JOIN "PrefixedNameDAO" AS "PrefixedNameDAO_1" ON "PrefixedNameDAO_1".database_id = "WorldEntityDAO_1"._name_id 
WHERE "TransportActionDAO".arm = :arm_1
```

1 result(s)
  Milk
